In [1]:
import warnings
warnings.filterwarnings('ignore')

Vanishing/Exploding Gradients Problems

Glorot and He Initialization

this is variance<br><br><br>
Glorot    ---  None, Tanh, Logistic, Softmax   ---    1 / fanavg<br>
He          ----    ReLU & variants         ---         2 / fanin<br>
LeCun        ---       SELU                ---         1 / fanin

By default, Keras uses Glorot initialization with a uniform distribution. You can change this to He initialization by setting kernel_initializer="he_uniform" or kernel_initializer="he_normal" when creating a layer, like this:

In [2]:
from tensorflow import keras
keras.layers.Dense(10,activation='relu',kernel_initializer='he_normal')

<Dense name=dense, built=False>

If you want He initialization with a uniform distribution, but based on fanavg rather
than fanin, you can use the VarianceScaling initializer like this:

In [3]:
he_avg_init = keras.initializers.VarianceScaling(scale=2,mode='fan_avg',
                                                 distribution='uniform')
keras.layers.Dense(10,activation='sigmoid',kernel_initializer=he_avg_init)

<Dense name=dense_1, built=False>

Nonsaturating Activation Functions

 LeakyReLUα
(z) = max(αz, z) 

 exponential linear unit (ELU)

ELU(z)   a(exp(z)-1)  if z<0  ,   z if z>=0

SELU(Scaled ELU)

To use the leaky ReLU activation function, you must create a LeakyReLU instance like this:

In [4]:
leaky_relu = keras.layers.LeakyReLU(alpha=0.2)
layer = keras.layers.Dense(10,activation=leaky_relu,
                           kernel_initializer='he_normal')

Batch Normalization

Implementing Batch Normalization with keras

In [5]:
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28,28]),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(30,activation='elu',kernel_initializer='he_normal'),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(40,activation='elu',kernel_initializer='he_normal'),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(10,activation='softmax')
])

In [6]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 784)            │         3,136 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 30)             │        23,550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 30)             │           120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 40)             │         1,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 40)             │           160 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │           410 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,616 (111.78 KB)

 Trainable params: 26,908 (105.11 KB)

 Non-trainable params: 1,708 (6.67 KB)

In [7]:
[(var.name, var.trainable) for var in model.layers[1].variables]

[('gamma', True),
 ('beta', True),
 ('moving_mean', False),
 ('moving_variance', False)]

In [17]:
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28,28]),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(30,kernel_initializer='he_normal',use_bias=False),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('elu'),
    keras.layers.Dense(20,activation='softmax')
])

Why Bias to False?<br> Because BatchNormalization just applies weight and bias to it.

<big>Gradient Clipping</big>

Another popular technique to lessen the exploding gradients problem is to simply
clip the gradients during backpropagation so that they never exceed some threshold.

It handles Only the gradient exploding problem---How I understood the concept

Mostly used in RNNs

In [19]:
optimizer = keras.optimizers.SGD(clipvalue=1.0)

Reusing Pretrained Layers

Transfer Learning with keras

TO load a Model: model_A  = keras.models.load_model("my_model_A.h5)

model_B_on_A = keras.models.Sequential(model_A.layers[-1])

Note that model_A and model_B_on_A now share some layers. When you train
model_B_on_A, it will also affect model_A. If you want to avoid that, you need to clone
model_A before you reuse its layers. To do this, you must clone model A’s architecture,
then copy its weights (since clone_model() does not clone the weights):

model_A_clone = keras.models.clone(model_A)<br>
model_A_clone = set_weights(model_A.get_weights())